# Mayne

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.Mayne)

class Mayne(LinearReferenceClock):
    def postprocess(self, x):
        """Gestational age in weeks (as published)."""
        return x



In [3]:
model = pya.models.Mayne()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = 'mayne'
model.metadata["data_type"] = 'methylation'
model.metadata["species"] = 'Homo sapiens'
model.metadata["year"] = 2017
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = "Mayne, Benjamin T., et al. \"Accelerated placental aging in early onset preeclampsia pregnancies identified by DNA methylation.\" Epigenomics 9.3 (2017): 279-289."
model.metadata["doi"] = "https://doi.org/10.2217/epi-2016-0103"
model.metadata["research_only"] = None
model.metadata["notes"] = "Placental DNA-methylation estimator of gestational age built from 62 CpG sites using penalized regression across pooled human placenta array datasets. Placentas from early-onset preeclampsia pregnancies show accelerated aging, with predicted gestational age exceeding chronological gestational age."
model.metadata["tissue"] = 'placenta (healthy singleton pregnancies)'
model.metadata["predicts"] = 'gestational age'
model.metadata["unit"] = 'weeks'
model.metadata["model_type"] = 'Elastic net'
model.metadata["platform"] = 'Illumina 27K/450K'
model.metadata["population"] = 'fetal/gestational (placental samples across pregnancy)'
model.metadata["journal"] = 'Epigenomics'
model.metadata["last_author"] = 'Tina Bianco‐Miotto'
model.metadata["n_features"] = 62
model.metadata["citations"] = 150
model.metadata["citations_date"] = '2026-07-05'

## Download clock dependencies

In [5]:
os.system(f"curl -sL -o coefficients.csv https://raw.githubusercontent.com/Duzhaozhen/OmniAge/c10fbe8cb92957520fbff1d55ae1def0691252e5/OmniAgePy/src/omniage/data/Mayne_GA.csv")

0

## Load features

In [6]:
df = pd.read_csv('coefficients.csv')
mask = df['probe'].astype(str).str.lower().isin(['intercept', '(intercept)'])
intercept_value = float(df.loc[mask, 'coef'].iloc[0]) if mask.any() else 0.0
coef_df = df.loc[~mask].reset_index(drop=True)
model.features = coef_df['probe'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['coef'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Mayne, Benjamin T., et al. "Accelerated placental aging in early '
             'onset preeclampsia pregnancies identified by DNA methylation." '
             'Epigenomics 9.3 (2017): 279-289.',
 'clock_name': 'mayne',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.2217/epi-2016-0103',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2017}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg12146151', 'cg16127845', 'cg17133388', 'cg13997435', 'cg12360736', 'cg07300408', 'cg11739626', 'cg00828602', 'cg21624359', 'cg22639768', 'cg11388238', 'cg06958211', 'cg19317715', 'cg20291222', 'cg03490200', 'cg26353877', 'cg22957381', 'cg19742789', 'cg02976574', 'cg16356956', 'cg26

## Basic test

In [13]:
torch.manual_seed(42)
input = torch.randn(10, len(model.features), dtype=float)
model.eval()
model.to(float)
pred = model(input)
pred

tensor([[ -39.5419],
        [  25.7659],
        [  44.2797],
        [  -0.9077],
        [  92.4899],
        [  86.2686],
        [-106.0699],
        [  76.7959],
        [  20.1038],
        [  21.3373]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: coefficients.csv
